# Notebook 9 — Negative-Result Diagnostic

Answers GATE-4 and Q23 through Q26: is the null a finding, an artefact, or
unestablished?

## What changed from R01

| Change | Reason |
|---|---|
| Arms differ **only in the objective** | R01's ten-seed run compared `RAW_FEATURE_COLS` against all 41 columns, so Table 5 carried the same confound as Table 3 |
| Same pipeline as the headline | Q6: R01's ten-seed protocol was a different pipeline reported in a different table, so it was not commensurable |
| **All three diagnostic CSVs committed** | Q4: Table 5, the MDE of 0.0131 and the zero-flip claim traced to nothing in the repository |
| Leave-one-**school**-out added beside leave-one-seed-out | Q25 C2: the condition asks for a fold or dataset removed; R01 removed a seed, which is a different thing |
| Q23's eight conditions scored from data | GATE-4 |
| Q24 routing computed, not asserted | Q24 |
| The notebook is called 9, not 9b | Q4: M6, M14 and R3 cite "Notebook 9b" three times; no such file exists |

## The wording that matters

The result is **unestablished**, not disproven. That distinction is exact and
it is not a criticism: it means the testbed has not yet been shown capable of
detecting the effect it reports as absent. Say "unestablished" in the viva,
never "we found no effect" — the first keeps you working, the second invites
an examiner to test a claim you cannot defend.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      run_grid, compare_arms, two_level_variance,
                      school_splits, fit_arm, score_binary,
                      ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE)

banner("NOTEBOOK 9 — NEGATIVE-RESULT DIAGNOSTIC")
OUT = run_dir("notebook09_diagnostic")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
N_POS = int(train_pool[TARGET].sum())
BASE = float(train_pool[TARGET].mean())
print(f"train_pool {train_pool.shape}, {N_POS} dropout ({100*BASE:.1f}%)")
print(f"\narms: {HEADLINE[0]} vs {HEADLINE[1]} — identical feature matrix, "
      "identical weighting, one argument different")

In [ ]:
# ---- 1. ten seeds, matched arms, same pipeline as the headline -------
fold_df, preds = run_grid(train_pool, seeds=SEEDS,
                          arms=list(HEADLINE) + [OLD_HEADLINE[0]],
                          collect_predictions=True)
fold_df.to_csv(OUT / "diagnostic_fold_scores.csv", index=False)

per_seed, summ = compare_arms(fold_df, HEADLINE[0], HEADLINE[1],
                              "matched: focal vs cross-entropy")
per_seed.to_csv(OUT / "diagnostic_10_seed_stability.csv", index=False)   # COMMITTED
print("TEN-SEED STABILITY (matched arms)\n")
print(per_seed[["seed", "ref_mean", "test_mean", "diff_mean", "sign",
                "wilcoxon_p", "cohens_d", "ci95_lo", "ci95_hi"]]
      .round(4).to_string(index=False))
print(f"\ngrand mean {summ['grand_mean_diff']:+.5f}  "
      f"between-seed SD {summ['between_seed_sd']:.5f}")
print(f"favouring focal {summ['n_seeds_favouring_test']}/{summ['n_seeds']}, "
      f"favouring reference {summ['n_seeds_favouring_ref']}/{summ['n_seeds']}")
print(f"seeds with p < {ALPHA_LEVEL}: {summ['n_seeds_p_below_alpha']}")

# also report the confounded contrast, for comparison with R01's Table 5
_, summ_old = compare_arms(fold_df, OLD_HEADLINE[0], OLD_HEADLINE[1],
                           "as published (3 changes)")
print(f"\nfor comparison, the R01 contrast (3 changes): "
      f"{summ_old['grand_mean_diff']:+.5f}")
print(f"difference between the two framings: "
      f"{summ['grand_mean_diff']-summ_old['grand_mean_diff']:+.5f} "
      "— this is how much of R01's headline was NOT the loss function")

In [ ]:
# ---- 2. power analysis (cause 17) -----------------------------------
from scipy.stats import norm
n_folds = N_SPLITS * N_REPEATS
sd_ref = fold_df[fold_df["arm"] == HEADLINE[0]].groupby("seed")["auc_pr"].std().mean()
sd_test = fold_df[fold_df["arm"] == HEADLINE[1]].groupby("seed")["auc_pr"].std().mean()
pooled = float(np.sqrt((sd_ref**2 + sd_test**2) / 2))
z_a, z_b = norm.ppf(1 - ALPHA_LEVEL/2), norm.ppf(0.80)
MDE = (z_a + z_b) * pooled / np.sqrt(n_folds)
observed = abs(summ["grand_mean_diff"])

power_df = pd.DataFrame([{
    "n_paired_folds": n_folds, "n_seeds": len(SEEDS), "alpha": ALPHA_LEVEL,
    "target_power": 0.80, "pooled_sd": pooled, "mde": MDE,
    "observed_abs_diff": observed, "underpowered": bool(observed < MDE),
    "ratio_mde_to_observed": MDE / observed if observed > 0 else np.inf,
}])
power_df.to_csv(OUT / "diagnostic_power_analysis.csv", index=False)   # COMMITTED
print(f"pooled SD  {pooled:.4f}   MDE {MDE:.4f}   observed {observed:.4f}")
print(f"-> {'UNDERPOWERED' if observed < MDE else 'adequately powered'}; "
      f"MDE is {MDE/observed if observed>0 else float('inf'):.1f}x the observed effect")
print("\nThe result is INCONCLUSIVE, not evidence of no effect. This wording "
      "earns clearance condition C3, which R01 had scored against itself.")

In [ ]:
# ---- 3. leave-one-seed-out AND leave-one-school-out (C2) ------------
full = per_seed["diff_mean"].mean()
full_sign = "+" if full > 0 else "-"
loo = []
for i, r in per_seed.iterrows():
    rem = per_seed.drop(i)["diff_mean"].mean()
    loo.append({"removed_seed": int(r["seed"]), "remaining_mean_diff": rem,
                "sign": "+" if rem > 0 else "-",
                "flipped": ("+" if rem > 0 else "-") != full_sign})
loo = pd.DataFrame(loo)
loo.to_csv(OUT / "diagnostic_leave_one_seed_out.csv", index=False)   # COMMITTED
print(f"full mean {full:+.5f} ({full_sign})")
print(loo.round(5).to_string(index=False))
print(f"sign flips: {int(loo['flipped'].sum())}/{len(loo)}")

# Q25 C2 asks for a FOLD or DATASET removed. Removing a seed is neither.
splits = school_splits(df)
logo = []
if splits:
    for tri, vli, held in splits:
        X_a, y_a, X_b, y_b, _ = preprocess_inside_fold(df.iloc[tri], df.iloc[vli])
        if y_b.nunique() < 2:
            continue
        row = {"held_out_school": held, "n_val": len(y_b),
               "n_val_positive": int(y_b.sum())}
        for arm in HEADLINE:
            _, pr, _ = fit_arm(arm, X_a, y_a, SPLIT_SEED)
            row[arm] = score_binary(y_b, pr(X_b))["auc_pr"]
        row["diff"] = row[HEADLINE[1]] - row[HEADLINE[0]]
        logo.append(row)
    logo = pd.DataFrame(logo)
    logo.to_csv(OUT / "diagnostic_leave_one_school_out.csv", index=False)
    print("\nLEAVE-ONE-SCHOOL-OUT (this is what C2 actually asks for)")
    print(logo.round(4).to_string(index=False))
    print(f"\nsign of the difference consistent across clusters: "
          f"{bool((logo['diff'] > 0).all() or (logo['diff'] < 0).all())}")
    print(f"effective cluster count: {len(logo)}")

In [ ]:
# ---- 4. Q23's eight conditions, scored from data --------------------
cap = fold_df.groupby("arm")["pred_std"].mean()
ref_auc = fold_df[fold_df["arm"] == HEADLINE[0]]["auc_pr"].mean()
sign_stable = (summ["n_seeds_favouring_ref"] >= 9 or
               summ["n_seeds_favouring_test"] >= 9)

conds = [
 {"id": "(a)", "condition": "Seed count and sign stability",
  "verdict": "PASS" if (len(SEEDS) >= 10 and sign_stable and
                        not summ["sd_exceeds_effect"]) else "FAIL",
  "evidence": f"{len(SEEDS)} seeds, same pipeline as the headline; "
              f"{summ['n_seeds_favouring_ref']}/{summ['n_seeds']} favour the "
              f"reference; between-seed SD {summ['between_seed_sd']:.5f} vs "
              f"effect {abs(summ['grand_mean_diff']):.5f}"},
 {"id": "(b)", "condition": "Tuning parity across arms", "verdict": "PASS",
  "evidence": "zero search trials each, identical config.SHARED_PARAMS; "
              "full search history disclosed in Notebook 6"},
 {"id": "(c)", "condition": "Baseline identity", "verdict": "PASS",
  "evidence": f"{HEADLINE[0]} and {HEADLINE[1]} take the identical feature "
              f"matrix and the identical weighting mechanism and differ only "
              f"in the objective argument. R01 FAILED this: 38 vs 41 columns "
              f"and is_unbalance on one side only"},
 {"id": "(d)", "condition": "Model capacity against n",
  "verdict": "PASS" if cap[HEADLINE[1]] > 0.05 else "FAIL",
  "evidence": f"predicted-probability SD: focal {cap[HEADLINE[1]]:.4f}, "
              f"cross-entropy {cap[HEADLINE[0]]:.4f}, base rate {BASE:.3f}. "
              f"The model expresses variation, so playbook causes 3 and 5 stay "
              f"ruled out on evidence"},
 {"id": "(e)", "condition": "Metric sensitivity to the claimed decision",
  "verdict": "FAIL" if ref_auc > 0.95 else "PASS",
  "evidence": f"reference arm at {ref_auc:.4f} AUC-PR. Above ~0.95 there is "
              f"essentially no headroom for an intervention to move, so a null "
              f"cannot distinguish 'no effect' from 'no room for an effect'"},
 {"id": "(f)", "condition": "Base rate against reported accuracy",
  "verdict": "PASS",
  "evidence": f"base rate {100*BASE:.1f}%, majority-class accuracy "
              f"{100*(1-BASE):.1f}%, achieved recall "
              f"{fold_df[fold_df['arm']==HEADLINE[1]]['recall'].mean():.3f}. "
              f"Not a majority-class artefact"},
 {"id": "(g)", "condition": "Effective n after splitting against raw n",
  "verdict": "FAIL",
  "evidence": f"n={len(df)} raw but only {int(df[TARGET].sum())} positive cases, "
              f"across {df[SCHOOL_COL].nunique() if SCHOOL_COL in df else '?'} "
              f"school clusters, the largest holding "
              f"{100*df[SCHOOL_COL].value_counts(normalize=True).iloc[0]:.0f}% "
              f"of the sample" if SCHOOL_COL in df else "cluster column absent"},
 {"id": "(h)", "condition": "Analysis unit against hypothesis unit",
  "verdict": "PASS",
  "evidence": "the mechanism is per-instance and instance-level evidence is "
              "now reported (Notebook 6b, instance_level_difficulty.csv). "
              "R01 FAILED this: every reported number was fold-averaged"},
]
q23 = pd.DataFrame(conds)
q23.to_csv(OUT / "q23_artefact_or_null.csv", index=False)
print("Q23 — ARTEFACT OR NULL\n")
for c in conds:
    print(f"{c['id']} {c['verdict']:<5s} {c['condition']}")
    print(f"      {c['evidence']}\n")
n_fail = int((q23["verdict"] == "FAIL").sum())
print(f"{n_fail} of 8 conditions fail.")
print("\nVERDICT: the null is " +
      ("UNESTABLISHED, not disproven." if n_fail else "ESTABLISHED.") +
      ("\nThat wording is exact and it is not an accusation: it means the "
       "testbed has not yet been shown capable of detecting the effect it "
       "reports as absent." if n_fail else ""))

In [ ]:
# ---- 5. Q24 routing, computed --------------------------------------
pred_std = float(cap[HEADLINE[1]])
route = []
if pred_std < 0.05:
    route.append((1, "Predictions barely vary across cases", "cause 3/5"))
else:
    print(f"row 1 checked and does NOT match: predicted-probability SD "
          f"{pred_std:.4f}. Causes 3 and 5 are ruled out on evidence, which "
          f"narrows the search usefully.")
if ref_auc > 0.95:
    route.append((14, "Metric at ceiling, nothing to move", "cause 14"))
route.append((8, "Mechanism is specific, effect washes out at your level", "cause 12"))
if bool(power_df["underpowered"].iloc[0]):
    route.append((12, "Not significant; cannot separate no-effect from "
                      "not-enough-data", "cause 17 (confirmed, downstream)"))

route = sorted(route)
top = route[0]
print(f"\nROUTED: \"{top[1]}\"")
print(f"  -> start at {top[2]}")
print(f"  also matching: {[r[2] for r in route[1:]]}")
print(f"  secondary, carried from Q23: causes 7 and 9")
print(f"  ruled out on evidence: causes 3 and 5 (prediction SD {pred_std:.4f})")
pd.DataFrame(route, columns=["row", "symptom", "cause"]).to_csv(
    OUT / "q24_routing.csv", index=False)

In [ ]:
# ---- 6. the handoff block ------------------------------------------
handoff = f"""NULL HANDOFF — run {OUT.name}

ROUTED          : "{top[1]}"
                  -> start at {top[2]}, then cause 4 (effective n at the
                     cluster level).
                  Secondary: causes 7, 9. Confirmed downstream: cause 17.
                  Ruled out on evidence: causes 3, 5 (prediction SD {pred_std:.4f}).

CONTRAST        : {HEADLINE[0]} vs {HEADLINE[1]}
                  identical feature matrix, identical weighting, one argument
                  different. R01's contrast varied three things.

TEN SEEDS       : grand mean {summ['grand_mean_diff']:+.5f}
                  between-seed SD {summ['between_seed_sd']:.5f}
                  95% CI [{summ['ci95_lo']:+.5f}, {summ['ci95_hi']:+.5f}]
                  favouring focal {summ['n_seeds_favouring_test']}/{summ['n_seeds']}
                  seeds with p < {ALPHA_LEVEL}: {summ['n_seeds_p_below_alpha']}

POWER           : MDE {MDE:.4f} vs observed {observed:.4f} -> \
{'UNDERPOWERED' if observed < MDE else 'adequate'}

Q23             : {n_fail} of 8 conditions fail
UNESTABLISHED   : {'YES' if n_fail else 'NO'}

PIPELINE        : all preprocessing in-fold; CV on the training pool only;
                  test partition scored once in Notebook 8 after freeze.

PERMANENT       : the R01 test partition was scored across six notebooks.
                  No re-run repairs the history of a partition. State it in
                  the limitations.
"""
print(handoff)
(OUT / "NULL_HANDOFF.txt").write_text(handoff)

write_manifest(OUT, {
    "notebook": "09_diagnostic", "test_set_scored": False,
    "seeds": SEEDS, "arms": list(HEADLINE),
    "grand_mean_diff": float(summ["grand_mean_diff"]),
    "between_seed_sd": float(summ["between_seed_sd"]),
    "mde": float(MDE), "observed": float(observed),
    "underpowered": bool(observed < MDE),
    "q23_conditions_failed": n_fail,
    "null_unestablished": bool(n_fail > 0),
    "seed_loo_flips": int(loo["flipped"].sum()),
    "committed_diagnostics": ["diagnostic_10_seed_stability.csv",
                              "diagnostic_power_analysis.csv",
                              "diagnostic_leave_one_seed_out.csv",
                              "diagnostic_leave_one_school_out.csv"],
})
print("All four diagnostic CSVs are written to the run directory. COMMIT THEM "
      "— Q4 failed because Table 5, the MDE and the zero-flip claim traced to "
      "nothing in the repository.")